# Multimodal Generative AI-Based E-Education System

Run Cells 1–5 in order on a Google Colab GPU runtime (NVIDIA Tesla T4, 15 GB VRAM) to install dependencies, download model checkpoints, and launch the Gradio Teacher/Student interface.


In [ ]:
# 1. Verify PyTorch and CUDA availability
import os
import sys

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ["PIP_NO_BUILD_ISOLATION"] = "1"

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Python {sys.version.split()[0]}")


In [ ]:
# 2. Install system packages and binary Python dependencies
import os
import shutil
import site
import subprocess
import sys

!apt-get update -qq
!apt-get install -y -qq ffmpeg espeak-ng espeak git libportaudio2

!pip install -q --only-binary=:all: \
    Pillow scipy pydub imageio-ffmpeg safetensors accelerate \
    yacs gradio python-docx python-pptx pandas scikit-image \
    tiktoken more-itertools tqdm diffusers transformers \
    onnxruntime pymatting pooch ctransformers edge-tts gTTS \
    addict lmdb pyyaml requests yapf kornia

!pip install -q --no-deps --no-build-isolation rembg openai-whisper future || true

# Install vision restoration packages from official repositories
!git clone -q --depth 1 https://github.com/rlabbe/filterpy.git /tmp/filterpy_repo 2>/dev/null || true
!git clone -q --depth 1 https://github.com/XPixelGroup/BasicSR.git /tmp/basicsr_repo 2>/dev/null || true
!git clone -q --depth 1 https://github.com/xinntao/facexlib.git /tmp/facexlib_repo 2>/dev/null || true
!git clone -q --depth 1 https://github.com/TencentARC/GFPGAN.git /tmp/gfpgan_repo 2>/dev/null || true

site_dir = site.getsitepackages()[0]
for repo_dir, pkg_name, ver_str in [
    ("/tmp/filterpy_repo", "filterpy", "1.4.5"),
    ("/tmp/basicsr_repo", "basicsr", "1.4.2"),
    ("/tmp/facexlib_repo", "facexlib", "0.3.0"),
    ("/tmp/gfpgan_repo", "gfpgan", "1.4.0"),
]:
    src_pkg = os.path.join(repo_dir, pkg_name)
    dst_pkg = os.path.join(site_dir, pkg_name)
    if os.path.exists(src_pkg):
        if os.path.exists(dst_pkg):
            shutil.rmtree(dst_pkg)
        shutil.copytree(src_pkg, dst_pkg)
        with open(os.path.join(dst_pkg, "version.py"), "w", encoding="utf-8") as vf:
            vf.write(f"__version__ = '{ver_str}'\n__gitsha__ = 'unknown'\nversion_info = ({ver_str.replace('.', ', ')})\n")

# Update deprecated torchvision import in BasicSR
deg_path = os.path.join(site_dir, "basicsr", "data", "degradations.py")
if os.path.exists(deg_path):
    with open(deg_path, "r", encoding="utf-8") as f:
        txt = f.read()
    txt = txt.replace(
        "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
        "from torchvision.transforms.functional import rgb_to_grayscale",
    )
    with open(deg_path, "w", encoding="utf-8") as f:
        f.write(txt)

# Install Coqui TTS on Python <= 3.12; configure lightweight TTS.api fallback bridge on Python 3.13+
if sys.version_info < (3, 13):
    subprocess.run(["pip", "install", "-q", "--only-binary=:all:", "coqui-tts==0.24.2"], check=False)

try:
    from TTS.api import TTS
    print("Coqui TTS is ready.")
except ImportError:
    print("Configuring TTS.api compatibility bridge for Python 3.13...")
    tts_pkg_dir = os.path.join(site_dir, "TTS")
    os.makedirs(tts_pkg_dir, exist_ok=True)
    with open(os.path.join(tts_pkg_dir, "__init__.py"), "w", encoding="utf-8") as f:
        f.write("# TTS compatibility package\n")
    with open(os.path.join(tts_pkg_dir, "api.py"), "w", encoding="utf-8") as f:
        f.write('''import os, subprocess
class _SynthesizerStub:
    pass

class TTS:
    def __init__(self, model_name="tts_models/en/ljspeech/tacotron2-DDC", progress_bar=False, gpu=False):
        self.model_name = model_name
        self.synthesizer = _SynthesizerStub()

    def tts_to_file(self, text, file_path="output.wav", speaker_wav=None, language="en", **kwargs):
        text = (text or "Hello.").strip()
        mp3_tmp = file_path + ".mp3"
        try:
            voice = "en-US-AriaNeural" if not speaker_wav else "en-US-ChristopherNeural"
            res = subprocess.run(["edge-tts", "--voice", voice, "--text", text, "--write-media", mp3_tmp], capture_output=True, timeout=30)
            if res.returncode == 0 and os.path.exists(mp3_tmp) and os.path.getsize(mp3_tmp) > 0:
                subprocess.run(["ffmpeg", "-y", "-i", mp3_tmp, "-ar", "22050", "-ac", "1", file_path], capture_output=True, timeout=30)
                if os.path.exists(mp3_tmp):
                    os.remove(mp3_tmp)
                if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
                    return file_path
        except Exception:
            pass
        try:
            from gtts import gTTS
            gTTS(text=text, lang=language or "en").save(mp3_tmp)
            subprocess.run(["ffmpeg", "-y", "-i", mp3_tmp, "-ar", "22050", "-ac", "1", file_path], capture_output=True, timeout=30)
            if os.path.exists(mp3_tmp):
                os.remove(mp3_tmp)
            if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
                return file_path
        except Exception:
            pass
        subprocess.run(["espeak-ng", "-w", file_path, text], capture_output=True)
        return file_path
''')

print("Dependencies installed.")


In [ ]:
# 3. Clone SadTalker and download pre-trained checkpoints
import glob
import os

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
!rm -rf /content/SadTalker
!git clone -q https://github.com/OpenTalker/SadTalker.git /content/SadTalker
!cd /content/SadTalker && bash scripts/download_models.sh
!wget -q -nc -O /content/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf

for st_file in glob.glob("/usr/local/lib/python*/dist-packages/safetensors/torch.py"):
    if os.path.exists(st_file):
        with open(st_file, "r", encoding="utf-8") as f:
            st_txt = f.read()
        if "torch.uint64" in st_txt:
            with open(st_file, "w", encoding="utf-8") as f:
                f.write(st_txt.replace("torch.uint64", "torch.long"))

print("Checkpoints ready.")


In [ ]:
# 4. Clone repository and apply SadTalker compatibility patches
import os
import sys

REPO_DIR = "/content/Multimodal-E-Education-System"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/AryanSingal/Multimodal-E-Education-System.git /content/Multimodal-E-Education-System
else:
    !cd /content/Multimodal-E-Education-System && git pull -q

src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from sadtalker_patches import patch_sadtalker_repo

patch_sadtalker_repo("/content/SadTalker")
print("SadTalker compatibility patches applied.")


In [ ]:
# 5. Initialize models and launch the Gradio E-Education System interface
import os
import sys

src_dir = "/content/Multimodal-E-Education-System/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from app import BASE_DATA_DIR, SHARED_FILES_DIR, TMP_DIR, create_complete_app, init_models

init_models()
demo = create_complete_app()
demo.launch(
    debug=True,
    share=True,
    allowed_paths=[
        TMP_DIR,
        BASE_DATA_DIR,
        "/tmp",
        os.path.join(SHARED_FILES_DIR, "notes"),
        os.path.join(SHARED_FILES_DIR, "assignments"),
        os.path.join(SHARED_FILES_DIR, "resources"),
    ],
    server_name="0.0.0.0",
    server_port=7860,
)
